<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_03_02_hpc_distributed_computing_sparklyr_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)

# Distributed Computing with {sparklyr}

The `sparklyr` package provides an R interface to Apache Spark, enabling distributed data processing and analysis with familiar R syntax. This notebook covers installation, connecting to Spark, data import, wrangling, visualization, modeling, and ML pipelines. The NYC Yellow Taxi dataset (January 2023) illustrates key workflows.

## Overview of sparklyr

**sparklyr** is an R interface for Apache Spark. It provides a **dplyr**-compatible backend so you can manipulate large-scale data in R while Spark executes work in a cluster.

**Key features**

- **Connection** — Connect to local or remote Spark (Databricks, AWS, Azure, YARN, Kubernetes, Livy, etc.).
- **dplyr integration** — Verbs like `filter()`, `group_by()`, `summarise()` translate to Spark SQL.
- **Machine learning** — MLlib via functions such as `ml_linear_regression()`, `ml_corr()`, and pipelines.
- **Extensibility** — Custom R code with `spark_apply()`, plus integration with **tidyr** and **ggplot2** (collect small results, then plot in R).

## Install and use sparklyr

On a normal R setup you can install from CRAN and optionally install a Spark distribution:

```r
install.packages("sparklyr")
library(sparklyr)
spark_install(version = "3.5")
```

This Colab notebook installs **sparklyr** from CRAN (see package cells below) and uses a downloaded **Apache Spark** binary; set `SPARK_HOME` if you run the notebook locally.


## Setup R in Python Runtime — Install {rpy2}

{rpy2} runs R code in Colab's Python runtime via the `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Drive if your data files live there. Edit `dataFolder` in the data-setup cell to point at your Parquet and CSV files (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install Apache Spark (Colab)

**sparklyr** needs a Spark installation and Java. Download a **Spark 3.5.x** distribution below (adjust `SPARK_VERSION` if needed). The Python cell sets `SPARK_HOME` for the process; R inherits it in many setups. **Local use:** install Spark yourself, set `SPARK_HOME`, and skip this cell.


In [ ]:
import os
import tarfile
import urllib.request

SPARK_VERSION = "3.5.4"
HADOOP_PROFILE = "hadoop3"
spark_dir = f"/content/spark-{SPARK_VERSION}-bin-{HADOOP_PROFILE}"
tar_name = f"spark-{SPARK_VERSION}-bin-{HADOOP_PROFILE}.tgz"
url = f"https://archive.apache.org/dist/spark/spark-{SPARK_VERSION}/{tar_name}"

if not os.path.isdir(spark_dir):
    tgz_path = f"/content/{tar_name}"
    if not os.path.isfile(tgz_path):
        print("Downloading Spark...")
        urllib.request.urlretrieve(url, tgz_path)
    print("Extracting...")
    with tarfile.open(tgz_path, "r:gz") as tf:
        tf.extractall("/content")
os.environ["SPARK_HOME"] = spark_dir
os.environ["PATH"] = os.pathsep.join([os.path.join(spark_dir, "bin"), os.environ.get("PATH", "")])
print("SPARK_HOME =", os.environ["SPARK_HOME"])


## Check and install required R packages

Install CRAN packages to Google Drive on Colab. **sparklyr** is installed from CRAN (unlike SparkR, which ships inside Spark's `R/lib` folder).


### Define required packages


In [ ]:
%%R
packages <- c(
  "sparklyr",
  "tidyverse",
  "arrow",
  "DBI",
  "ggplot2"
)


### Install missing packages (Google Drive library)


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages


### Check loaded packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


## Connecting to Spark with sparklyr

Use `spark_connect()` to attach to a cluster. Examples (reference only):

```r
sc <- spark_connect(master = "local")
spark_disconnect(sc)

# Other masters
# sc <- spark_connect(master = "yarn")
# sc <- spark_connect(master = "mesos://host:port")
# sc <- spark_connect(master = "k8s://https://server")
# sc <- spark_connect(master = "http://server/livy", method = "livy")
```

**Configuration** — adjust driver and executor memory (Colab: use modest values such as 2–4 GB).

```r
config <- spark_config()
config$spark.driver.memory <- "4g"
config$spark.executor.memory <- "4g"
sc <- spark_connect(master = "local", version = "3.5.0", config = config)
```

## Data import (reference)

- **CSV** — `spark_read_csv(..., header = TRUE, infer_schema = TRUE, ...)`
- **JSON** — `spark_read_json()`
- **Parquet** — `spark_read_parquet()`
- **Delta** — `spark_read_delta()`
- **Table reference** — `dplyr::tbl(sc, dbplyr::in_catalog("catalog", "schema", "table"))`
- **Copy R data to Spark** — `dplyr::copy_to(dest, df, name)` (with Arrow for faster transfer when available)

See the [sparklyr cheatsheet](https://rstudio.github.io/cheatsheets/html/sparklyr.html) for data import and ML pipelines.

## dplyr, tidyr, transformers, ML, pipelines, spark_apply

- **dplyr verbs** on Spark tables translate to Spark SQL; chain with `%>%` and `collect()` when you need an R tibble.
- **tidyr** — `pivot_longer`, `pivot_wider`, `nest`/`unnest`, etc., where supported for your Spark version.
- **Feature transformers** — `ft_binarizer()`, `ft_bucketizer()`, `ft_vector_assembler()`, `ft_standard_scaler()`, and many others in **sparklyr**.
- **Modeling** — e.g. `ml_linear_regression()`, `ml_logistic_regression()`, `ml_random_forest_*`, `ml_kmeans()`, `ml_corr()`, `ml_evaluate()`, etc.
- **Pipelines** — `ml_pipeline()`, `ml_fit()`, `ml_save()`, `ml_read()` for portable Spark ML pipelines.
- **Distributed R** — `spark_apply()` runs arbitrary R code on partitions (use for embarrassingly parallel tasks).

The sections below run end-to-end examples on NYC taxi data.


## Basic data processing with sparklyr

Set `SPARK_HOME` if it is not inherited from Python (Colab path should match the Spark install cell). Use **moderate** memory on Colab; increase locally if you have RAM.


In [ ]:
%%R
sh <- Sys.getenv("SPARK_HOME")
if (nchar(sh) < 1) Sys.setenv(SPARK_HOME = "/content/spark-3.5.4-bin-hadoop3")
Sys.setenv(SPARK_HOME = Sys.getenv("SPARK_HOME"))

config <- spark_config()
# Colab-friendly defaults; increase for local heavy runs
config$spark.driver.memory <- "4g"
config$spark.executor.memory <- "4g"

sc <- spark_connect(
  master = "local",
  version = "3.5",
  config = config
)


### Data paths (Colab vs local)


In [ ]:
%%R
# Colab: uncomment and set your Drive folder
# dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
# dataFolder <- paste0(sub("/$", "", dataFolder), "/")

# Local / repo (edit if needed)
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
trip_data_path <- paste0(dataFolder, "yellow_tripdata_2023-01.parquet")
zone_lookup_path <- paste0(dataFolder, "taxi_zone_lookup.csv")

trips <- spark_read_parquet(sc, "trips", trip_data_path)
zones <- spark_read_csv(sc, "zones", zone_lookup_path, header = TRUE, infer_schema = TRUE)


### Exploring the data


In [ ]:
%%R
print(sdf_schema(trips))
print(sdf_schema(zones))
trips %>% head(5) %>% collect()


#### Registering Spark tables / temporary views


In [ ]:
%%R
trips %>% sdf_register("trips")
zones %>% sdf_register("zones")

trip_count <- dbGetQuery(sc, "SELECT COUNT(*) AS total_trips FROM trips")
print(trip_count)


In [ ]:
%%R
trip_count_dplyr <- trips %>%
  dplyr::summarise(total_trips = n()) %>%
  collect()
print(trip_count_dplyr)


#### Summary statistics


In [ ]:
%%R
summary_trips <- trips %>%
  dplyr::summarise(
    avg_distance = mean(trip_distance, na.rm = TRUE),
    avg_fare = mean(total_amount, na.rm = TRUE),
    max_distance = max(trip_distance, na.rm = TRUE)
  )
collect(summary_trips)


#### Joining datasets


In [ ]:
%%R
joined_trips <- trips %>%
  left_join(zones, by = c("PULocationID" = "LocationID")) %>%
  dplyr::rename(pickup_borough = Borough, pickup_zone = Zone)

joined_trips %>% sdf_register("joined_trips")


## Data analysis


### Average trip distance by borough


In [ ]:
%%R
avg_distance_borough <- DBI::dbGetQuery(sc, "
  SELECT pickup_borough, AVG(trip_distance) AS avg_distance
  FROM joined_trips
  GROUP BY pickup_borough
  ORDER BY avg_distance DESC
")
collect(avg_distance_borough)


### Total fare amount by payment type


In [ ]:
%%R
total_fare_payment <- trips %>%
  dplyr::group_by(payment_type) %>%
  dplyr::summarise(total_fare = sum(total_amount, na.rm = TRUE)) %>%
  dplyr::arrange(desc(total_fare))
collect(total_fare_payment)


### Average fare by pickup zone


In [ ]:
%%R
avg_fare_by_zone <- DBI::dbGetQuery(sc, "
  SELECT pickup_zone, AVG(total_amount) AS avg_fare, COUNT(*) AS trip_count
  FROM joined_trips
  WHERE total_amount > 0 AND trip_distance > 0
  GROUP BY pickup_zone
  HAVING trip_count > 100
  ORDER BY avg_fare DESC
")
collect(avg_fare_by_zone)


### Trip duration analysis


In [ ]:
%%R
trips_with_duration <- trips %>%
  filter(tpep_dropoff_datetime > tpep_pickup_datetime) %>%
  mutate(duration_minutes = (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60) %>%
  select(tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, duration_minutes)

trips_with_duration %>% sdf_register("trips_with_duration")

avg_duration <- trips_with_duration %>%
  dplyr::group_by(passenger_count) %>%
  dplyr::summarise(avg_duration = mean(duration_minutes, na.rm = TRUE)) %>%
  dplyr::arrange(passenger_count) %>%
  collect()
print(avg_duration)


In [ ]:
%%R
peak_hours <- trips %>%
  mutate(pickup_hour = hour(tpep_pickup_datetime)) %>%
  dplyr::group_by(pickup_hour) %>%
  dplyr::summarise(trip_count = n()) %>%
  arrange(desc(trip_count))
collect(peak_hours)


### Visualization: popular pickup boroughs


In [ ]:
%%R
popular_boroughs <- joined_trips %>%
  dplyr::group_by(pickup_borough) %>%
  dplyr::summarise(trip_count = n()) %>%
  dplyr::arrange(desc(trip_count))

popular_boroughs_local <- collect(popular_boroughs)

ggplot(popular_boroughs_local, aes(x = reorder(pickup_borough, -trip_count), y = trip_count)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  labs(title = "Most Popular Pickup Boroughs (Jan 2023)", x = "Borough", y = "Trip Count") +
  theme_minimal()


### Correlation analysis


In [ ]:
%%R
trip_data <- trips %>%
  select(trip_distance, fare_amount, tip_amount) %>%
  na.omit()

trip_data %>% head(5) %>% collect()

cor_trip_fare <- ml_corr(trip_data, columns = c("trip_distance", "fare_amount"))[1, 2]
cor_trip_tip <- ml_corr(trip_data, columns = c("trip_distance", "tip_amount"))[1, 2]
cor_fare_tip <- ml_corr(trip_data, columns = c("fare_amount", "tip_amount"))[1, 2]

cat("Distributed Pearson Correlation Matrix:\n")
cor_matrix <- matrix(
  c(1, cor_trip_fare, cor_trip_tip,
    cor_trip_fare, 1, cor_fare_tip,
    cor_trip_tip, cor_fare_tip, 1),
  nrow = 3, byrow = TRUE,
  dimnames = list(
    c("trip_distance", "fare_amount", "tip_amount"),
    c("trip_distance", "fare_amount", "tip_amount")
  )
)
print(cor_matrix)


### Simple linear regression

Predict `fare_amount` from `trip_distance` only.


In [ ]:
%%R
simple_model <- trip_data %>%
  ml_linear_regression(fare_amount ~ trip_distance, reg_param = 0.01, max_iter = 100)
summary(simple_model)


### Multi-variable linear regression

Predict `fare_amount` from `trip_distance` and `tip_amount`.


In [ ]:
%%R
multi_model <- trip_data %>%
  ml_linear_regression(fare_amount ~ trip_distance + tip_amount, reg_param = 0.01, max_iter = 100)
summary(multi_model)


### Disconnect Spark (optional)

Release resources when finished.


In [ ]:
%%R
spark_disconnect(sc)


## Summary and conclusions

This notebook showed how to use **sparklyr** with Apache Spark: `spark_connect()`, `spark_read_parquet()` / `spark_read_csv()`, **dplyr** and **DBI** workflows, joins and aggregations on NYC taxi data, duration and peak-hour summaries, **ggplot2** after `collect()`, distributed correlations with `ml_corr()`, and linear regression with `ml_linear_regression()`. Tune memory and Spark version to match your environment; avoid collecting large tables into R.

## Resources

1. [sparklyr.ai](https://sparklyr.ai/) — official documentation  
2. [Mastering Spark with R](https://therinspark.com/)  
3. [Posit Community — sparklyr](https://community.rstudio.com/c/sparklyr)  
4. [DataCamp — Big Data with R and Spark](https://www.datacamp.com/courses/big-data-with-r-and-spark)  
5. Search **sparklyr Edgar Ruiz** on YouTube for video introductions.
